### Drivers: Bronze to Silver
Clean and transform raw drivers data from `formula1_incr.bronze.drivers` into `formula1_incr.silver.drivers`.

#### Setup
- `01.environment-config` → loads catalog name, bronze/silver schema names
- `03.silver_helpers` → loads the `write_to_silver()` function we use to save data

In [0]:
%run ../00-common/01.environment-config 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')


In [0]:
%run ../00-common/03.silver_helpers

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.drivers'
silver_table = f'{catalog_name}.{silver_schema}.drivers'

#### Read Bronze
- Read raw data from the bronze table filtered by `batch_id`

In [0]:
drivers_df = spark.read.table(bronze_table).filter(col('batch_id') == v_batch_id)
display(drivers_df)

#### Drop Columns

#### Drop Columns
- Remove `url` column — not needed for analysis

In [0]:
drivers_drop_df = drivers_df.drop('url')

#### Rename Columns
- Convert camelCase to snake_case for consistency

In [0]:
from pyspark.sql import functions as F
drivers_renamed_df = drivers_drop_df.withColumnRenamed('driverId', 'driver_id').withColumnRenamed('dateOfBirth', 'date_of_birth')                    

In [0]:
display(drivers_renamed_df)

#### Concatenate Names
- Combine `name.givenName` and `name.familyName` into one `driver_name` column

In [0]:
from pyspark.sql.functions import *
drivers_concatenate_df =(
    drivers_renamed_df
    .withColumn('driver_name',
              initcap(concat_ws(' ', col('name.givenName'), F.col('name.FamilyName'))))
    .drop('name')
)
display(drivers_concatenate_df)

#### Remove Duplicates
- Keep one row per driver using `dropDuplicates()`

In [0]:
drivers_distinct_df = drivers_concatenate_df.dropDuplicates(['driver_id'])


#### Title Case
- Apply `initcap()` to `nationality` so it looks clean

In [0]:
drivers_final_df = (drivers_distinct_df.withColumn('nationality', initcap(col('nationality'))))


#### Write to Silver
- If the silver table doesn't exist yet, it creates it from scratch
- If it already exists, it merges new/updated rows using `write_to_silver()` (insert new, update changed)

In [0]:
write_to_silver(
    input_df=drivers_final_df,
    target_table=silver_table,
    merge_condition='t.driver_id = s.driver_id',
    columns_to_update=[
        'driver_id',
        'date_of_birth',
        'nationality',
        'ingestion_timestamp',
        'source_file',
        'batch_id',
        'driver_name'
    ]
)

In [0]:
spark.read.table(silver_table).display() 